<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 6 · Martes — SimpleImputer y Guardar Modelos</h1>
<h3>Manejo de nulos dentro del pipeline + persistencia con joblib</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final vas a poder:

1. Usar **`SimpleImputer`** para manejar nulos dentro de un pipeline.
2. Conocer las **4 estrategias de imputación** y cuándo usar cada una.
3. Construir un pipeline **completo** con: imputación + escalado + encoding + modelo.
4. **Guardar y cargar modelos** con `joblib` (persistencia profesional).
5. Hacer **predicciones en producción** desde un modelo guardado.
6. Resolver **3 ejercicios** prácticos.

> Requisito previo: la clase de ayer (Pipelines y ColumnTransformer).

# 1. El problema de los nulos en producción

Hasta ahora usábamos `df.dropna()` o `df.fillna(0)` **antes** de entrenar. El problema:

🚨 **En producción, los datos NUEVOS también pueden tener nulos**. Si tu pipeline no sabe qué hacer con nulos, **se rompe**.

Ejemplo real: un cliente nuevo no completó su `edad` en el formulario. Tu modelo en producción **falla** porque no sabe imputar.

**Solución:** poner la imputación DENTRO del pipeline con **`SimpleImputer`**. Así, cualquier dato nuevo se imputa automáticamente con la estrategia que aprendió del train.

# 2. `SimpleImputer` — las 4 estrategias

`SimpleImputer` recibe un parámetro `strategy=` con 4 opciones:

| Estrategia | ¿Qué hace? | Cuándo usar |
|---|---|---|
| `'mean'` | Rellena con la **media** de la columna | Numérica con distribución simétrica |
| `'median'` | Rellena con la **mediana** | Numérica con outliers (más robusta) |
| `'most_frequent'` | Rellena con el **valor más común** (moda) | Funciona en numéricas Y categóricas |
| `'constant'` | Rellena con un valor fijo (`fill_value=`) | Cuando quieres un marcador explícito como 0 o `'Unknown'` |

### 🧠 Regla práctica

- 🔢 **Numéricas → `median`** (más robusta ante outliers)
- 🏷️ **Categóricas → `most_frequent` o `constant` con `'Unknown'`**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# DataFrame con nulos
df = pd.DataFrame({
    'edad':    [25, 30, np.nan, 45, 28, np.nan, 50],
    'ingreso': [1000, 1500, 2000, np.nan, 1200, 1800, np.nan],
    'ciudad':  ['Santiago', 'Valparaíso', np.nan, 'Santiago', 'Concepción', 'Santiago', np.nan]
})
print('Antes:')
print(df)
print(f'\nNulos: {df.isnull().sum().sum()}')

In [ ]:
# Imputer para numéricas — mediana
imp_num = SimpleImputer(strategy='median')
df[['edad', 'ingreso']] = imp_num.fit_transform(df[['edad', 'ingreso']])

print('Después de imputar numéricas con mediana:')
print(df)
print(f'\nValores aprendidos por el imputer: {imp_num.statistics_}')

In [ ]:
# Imputer para categórica — most_frequent (o constant con 'Unknown')
imp_cat = SimpleImputer(strategy='most_frequent')
df[['ciudad']] = imp_cat.fit_transform(df[['ciudad']])

print('Después de imputar categórica con most_frequent:')
print(df)
print(f'\nValor más frecuente aprendido: {imp_cat.statistics_}')
print(f'\nNulos restantes: {df.isnull().sum().sum()} ✅')

# 3. Pipeline COMPLETO — imputer + scaler + encoder + modelo

Ahora vamos a juntar todo lo aprendido. Pipeline profesional:

```
Numéricas:    SimpleImputer(median) → StandardScaler
Categóricas:  SimpleImputer(most_frequent) → OneHotEncoder
                            ↓
                        Modelo final
```

Esto se construye con un **Pipeline anidado dentro de un ColumnTransformer dentro de otro Pipeline**. Suena complejo pero es elegante.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn import set_config

set_config(display='diagram')

# Vamos a usar penguins — tiene nulos REALES en varias columnas
pen = sns.load_dataset('penguins')   # ojo: NO hago dropna()
print(f'Shape: {pen.shape}')
print(f'Nulos por columna:')
print(pen.isnull().sum())

In [ ]:
X = pen.drop(columns=['body_mass_g'])
y = pen['body_mass_g'].fillna(pen['body_mass_g'].median())  # también imputamos el target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipelines internos para cada tipo de columna
pipe_num = make_pipeline(SimpleImputer(strategy='median'),  StandardScaler())
pipe_cat = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))

preprocesador = make_column_transformer(
    (pipe_num, make_column_selector(dtype_include=np.number)),
    (pipe_cat, make_column_selector(dtype_include=object))
)

# Pipeline FINAL con el modelo
modelo = make_pipeline(preprocesador, LinearRegression())
modelo

In [ ]:
# Una sola línea — entrena imputación + scaling + encoding + modelo
modelo.fit(X_train, y_train)

print(f'R² test: {modelo.score(X_test, y_test):.4f}')
print(f'\n💡 Nota: aunque X_train tiene NULOS, el pipeline los maneja AUTOMÁTICAMENTE.')
print(f'         Lo mismo va a pasar si llega un dato nuevo en producción con nulos.')

# 4. Guardar y cargar modelos con `joblib`

## ¿Por qué necesitamos guardar el modelo?

Entrenar un modelo puede tardar **horas**. No quieres re-entrenarlo cada vez que tu app arranca. La solución es **persistir** el modelo entrenado en un archivo, y cargarlo cuando lo necesites.

### `joblib` vs `pickle`

| | `pickle` | `joblib` |
|---|---|---|
| Origen | Stdlib de Python | Recomendado por sklearn |
| Velocidad | Lenta con numpy | **Más rápida con numpy** |
| Uso | Genérico | Optimizado para sklearn |

👉 **Usa `joblib`** para modelos de sklearn — es el estándar profesional.

In [ ]:
import joblib

# Guardar el pipeline completo (¡guarda TODO: imputers, scaler, encoder, modelo!)
joblib.dump(modelo, 'modelo_pinguinos.joblib')

print('✅ Modelo guardado en "modelo_pinguinos.joblib"')
print('   Este archivo se puede mover a producción, otro computador, otro servidor.')

In [ ]:
# CARGAR el modelo desde disco (simulamos que estamos en otro script/servidor)
modelo_cargado = joblib.load('modelo_pinguinos.joblib')

# Verificamos que da el mismo resultado
print(f'R² del modelo cargado: {modelo_cargado.score(X_test, y_test):.4f}')
print('👉 Es exactamente el mismo que antes — está intacto.')

In [ ]:
# Simulamos un dato NUEVO llegando a producción (con nulos!)
pinguino_nuevo = pd.DataFrame([{
    'species':           'Adelie',
    'island':            'Torgersen',
    'bill_length_mm':    39.1,
    'bill_depth_mm':     np.nan,         # ← NULO en producción
    'flipper_length_mm': 181.0,
    'sex':               np.nan          # ← NULO en producción
}])

print('Pingüino nuevo (con nulos):')
print(pinguino_nuevo)

prediccion = modelo_cargado.predict(pinguino_nuevo)
print(f'\n🐧 Peso predicho: {prediccion[0]:.0f} gramos')
print('   ✅ El modelo manejó los nulos y predijo sin problema.')

# 5. Buenas prácticas para producción

| Práctica | Por qué |
|---|---|
| **Guardar SIEMPRE el pipeline completo**, no solo el modelo | El modelo solo no sabe cómo escalar/codificar |
| **Versionar el archivo del modelo** (ej. `modelo_v1.joblib`) | Para volver atrás si algo falla |
| **Guardar también la versión de sklearn** usada | Cambios de versión pueden romper modelos |
| **Validar el output** antes de usarlo en producción | Detectar predicciones absurdas |
| **handle_unknown='ignore'** en OneHotEncoder | Para no romper si llega una categoría nueva |

---
# 🏋️ Ejercicios prácticos

## Ejercicio 1 — Pipeline con imputación sobre `mpg`

**Tarea:**

1. Cargar `sns.load_dataset('mpg')` — **NO** hagas dropna (tiene nulos reales).
2. Verificar que hay nulos en alguna columna numérica.
3. Target: `mpg`. Features: TODAS las demás (incluyendo `origin` categórica). Excluir `name`.
4. Construir un Pipeline con:
   - Para numéricas: `SimpleImputer(median)` + `StandardScaler`
   - Para categóricas: `SimpleImputer(most_frequent)` + `OneHotEncoder`
   - Modelo: `LinearRegression`
5. Train/test 80/20, entrenar y reportar R².

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector

mpg = sns.load_dataset('mpg').drop(columns=['name'])
print('Nulos por columna:'); print(mpg.isnull().sum())

X = mpg.drop(columns=['mpg'])
y = mpg['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe_num = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
pipe_cat = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))

preproc = make_column_transformer(
    (pipe_num, make_column_selector(dtype_include=np.number)),
    (pipe_cat, make_column_selector(dtype_include=object))
)

modelo = make_pipeline(preproc, LinearRegression()).fit(X_train, y_train)
print(f'\nR² test: {modelo.score(X_test, y_test):.4f}')
```
</details>

## Ejercicio 2 — Guardar y cargar un modelo

Vamos a simular el flujo completo de producción.

**Tarea:**

1. Usar el modelo entrenado del Ejercicio 1 (o entrenar uno nuevo).
2. **Guardarlo** en `modelo_mpg.joblib` con `joblib.dump()`.
3. **Cargarlo** en otra variable con `joblib.load()`.
4. Verificar que el R² del modelo cargado es **idéntico** al original.
5. Crear un **auto inventado** como DataFrame (con todas las features) — al menos una con nulo.
6. Hacer la predicción del consumo en mpg.

**Auto sugerido:**
```python
auto = pd.DataFrame([{
    'cylinders': 4, 'displacement': 150, 'horsepower': np.nan,
    'weight': 2800, 'acceleration': 16.0, 'model_year': 78, 'origin': 'usa'
}])
```

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
import joblib

# Guardar
joblib.dump(modelo, 'modelo_mpg.joblib')
print('✅ Guardado')

# Cargar
modelo_cargado = joblib.load('modelo_mpg.joblib')
print(f'R² original: {modelo.score(X_test, y_test):.4f}')
print(f'R² cargado:  {modelo_cargado.score(X_test, y_test):.4f}  → ¡idéntico!')

# Predicción de auto nuevo (con nulo)
auto = pd.DataFrame([{
    'cylinders': 4, 'displacement': 150, 'horsepower': np.nan,
    'weight': 2800, 'acceleration': 16.0, 'model_year': 78, 'origin': 'usa'
}])
print(f'\nConsumo predicho: {modelo_cargado.predict(auto)[0]:.2f} mpg')
```
</details>

## Ejercicio 3 — Comparar estrategias de imputación

¿Realmente importa qué estrategia de imputación elegimos? Lo verificamos con datos.

**Tarea:**

1. Cargar `data/housing.csv` (NO hacer dropna).
2. Target: `median_house_value`. Features: TODAS las demás.
3. Construir **3 pipelines** idénticos excepto por la estrategia de imputación de numéricas:
   - Pipeline A: `SimpleImputer(strategy='mean')`
   - Pipeline B: `SimpleImputer(strategy='median')`
   - Pipeline C: `SimpleImputer(strategy='constant', fill_value=0)`
4. Todos con `StandardScaler` + `OneHotEncoder` para categórica + `LinearRegression`.
5. Train/test 80/20.
6. Reportar R² de cada uno y discutir: ¿cambia mucho?

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector

df = pd.read_csv('data/housing.csv')
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def crear_pipeline(estrategia, fill_value=None):
    imp = SimpleImputer(strategy=estrategia, fill_value=fill_value)
    pipe_num = make_pipeline(imp, StandardScaler())
    pipe_cat = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))
    preproc = make_column_transformer(
        (pipe_num, make_column_selector(dtype_include=np.number)),
        (pipe_cat, make_column_selector(dtype_include=object))
    )
    return make_pipeline(preproc, LinearRegression())

for nombre, est, fv in [('mean', 'mean', None), ('median', 'median', None), ('constant=0', 'constant', 0)]:
    m = crear_pipeline(est, fv).fit(X_train, y_train)
    print(f'Imputación {nombre:12s} → R² = {m.score(X_test, y_test):.4f}')

# 👉 En este dataset los 3 dan resultados muy similares (housing tiene pocos nulos).
#    En datasets con MUCHOS nulos, la estrategia sí marca diferencia notable.
```
</details>

---
## 📌 Cierre del día

Hoy aprendimos:

- ✅ **`SimpleImputer`** y sus 4 estrategias (`mean`, `median`, `most_frequent`, `constant`)
- ✅ Cómo incluir imputación **dentro** del pipeline → nulos manejados en producción
- ✅ Pipeline profesional completo: `Imputer + Scaler + Encoder + Modelo`
- ✅ **`joblib`** para guardar/cargar modelos
- ✅ Buenas prácticas de producción

### 🔜 Mañana — Miércoles 27

- **Random Forest y Bagging** — combinar muchos árboles para mejorar predicciones
- Feature Importance — qué variables pesan más para el modelo

Nos vemos 🚀